In [4]:
from pydantic_settings import BaseSettings, SettingsConfigDict
import asyncio
BASE_DIR = Path.cwd().parent   # notebook in src/

class Settings(BaseSettings):
    entsoe_api_key: str
    
    model_config = SettingsConfigDict(
        env_file=BASE_DIR / ".env",
        extra="ignore"
    )

settings = Settings()

In [7]:
import aiohttp
import pandas as pd
from pathlib import Path


BASE_URL = "https://web-api.tp.entsoe.eu/api"
SWISS_DOMAIN = "10YCH-SWISSGRIDZ"
BASE_PATH = Path

async def fetch_entsoe_xml_year_by_year(
    api_key: str,
    start_year: int,
    end_year: int,
    save_dir: str | Path = Path.cwd().parent / "data" / "raw_xml",
):
    """
    Download Swiss ENTSO-E total load XML year by year.

    Example:
        await fetch_entsoe_xml_year_by_year(
            api_key=settings.entsoe_api_key,
            start_year=2020,
            end_year=2026
        )
    """

    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    async with aiohttp.ClientSession() as session:

        for year in range(start_year, end_year + 1):

            start = f"{year}01010000"
            end = f"{year + 1}01010000"

            params = {
                "documentType": "A65",
                "processType": "A16",
                "outBiddingZone_Domain": SWISS_DOMAIN,
                "periodStart": start,
                "periodEnd": end,
                "securityToken": api_key,
            }

            file_path = save_path / f"swiss_load_{year}.xml"

            async with session.get(BASE_URL, params=params) as response:
                response.raise_for_status()

                xml_text = await response.text()

                file_path.write_text(xml_text, encoding="utf-8")

                print(f"Saved: {file_path}")
                
await fetch_entsoe_xml_year_by_year(
    api_key=settings.entsoe_api_key,
    start_year=2020,
    end_year=2026
)

Saved: /home/alvard/projects/load_forecasting/data/raw_xml/swiss_load_2020.xml
Saved: /home/alvard/projects/load_forecasting/data/raw_xml/swiss_load_2021.xml
Saved: /home/alvard/projects/load_forecasting/data/raw_xml/swiss_load_2022.xml
Saved: /home/alvard/projects/load_forecasting/data/raw_xml/swiss_load_2023.xml
Saved: /home/alvard/projects/load_forecasting/data/raw_xml/swiss_load_2024.xml
Saved: /home/alvard/projects/load_forecasting/data/raw_xml/swiss_load_2025.xml
Saved: /home/alvard/projects/load_forecasting/data/raw_xml/swiss_load_2026.xml
